# Demo phát hiện SQL Injection — Nhánh 1 + Nhánh 2

Load 2 model đã train (`models/nhanh1_v1/`, `models/nhanh2_v1/`), nhập một câu query/payload,
trả về kết quả kết hợp: nhãn Nhánh 1 (loại tấn công), điểm bất thường Nhánh 2, và verdict cuối cùng.

**Verdict logic (đơn giản hoá, chưa phải Bộ xử lý trung tâm đầy đủ):**
- Nhánh 1 = tấn công → **BLOCK**
- Nhánh 1 = normal + Nhánh 2 bất thường → **OVERKILL** (cần Admin xác nhận)
- Nhánh 1 = normal + Nhánh 2 bình thường → **ALLOW**


## 1. Load model

In [1]:
import sys
sys.path.insert(0, '..')

import joblib
import numpy as np
from src.preprocessing.canonicalize import canonicalize
from src.preprocessing.multiclass_tagger import LABEL_NAMES
from src.preprocessing.statistical_features import extract_statistical_features
from src.models.nhanh2_anomaly import AnomalyDetector
from src.utils import load_config

cfg = load_config('../configs/config.yaml')

# Nhanh 1: TF-IDF + Logistic Regression
nhanh1_vectorizer = joblib.load('../models/nhanh1_v1/vectorizer.joblib')
nhanh1_clf = joblib.load('../models/nhanh1_v1/model.joblib')

# Nhanh 2: One-Class SVM (anomaly detector)
nhanh2_detector = AnomalyDetector.load('../models/nhanh2_v1')

print('Da load xong Nhanh 1 (TF-IDF+LogReg) va Nhanh 2 (OCSVM)')


2026-07-21 10:14:51,997 | INFO     | src.models.nhanh2_anomaly | Model loaded from ../models/nhanh2_v1


Da load xong Nhanh 1 (TF-IDF+LogReg) va Nhanh 2 (OCSVM)


## 2. Hàm `detect()` — nhập query, trả kết quả

In [2]:
def detect(query: str, anomaly_threshold: float = None) -> dict:
    """Chay 1 query qua Nhanh 1 + Nhanh 2, tra ve ket qua ket hop."""
    canonical = canonicalize(query, max_decode_iterations=cfg.get_path('preprocessing.max_decode_iterations', 3))
    text = canonical.query_canonical

    # --- Nhanh 1: supervised multiclass ---
    X1 = nhanh1_vectorizer.transform([text])
    label = int(nhanh1_clf.predict(X1)[0])
    proba = nhanh1_clf.predict_proba(X1)[0]
    label_name = LABEL_NAMES[label]
    nhanh1_confidence = float(proba[list(nhanh1_clf.classes_).index(label)])

    # --- Nhanh 2: anomaly detection (chi dua tren dac trung thong ke) ---
    feats = extract_statistical_features(text)
    X2 = np.array([feats.as_list()])
    anomaly_score = float(nhanh2_detector.score(X2)[0])
    is_anomaly = bool(nhanh2_detector.anomaly_flags(X2, threshold=anomaly_threshold)[0])

    # --- Verdict gop (logic don gian, chua phai Bo xu ly trung tam day du) ---
    if label != 0:  # 0 = normal
        verdict = 'BLOCK'
    elif is_anomaly:
        verdict = 'OVERKILL'
    else:
        verdict = 'ALLOW'

    return {
        'query_raw': query,
        'query_canonical': text,
        'has_comment_marker': canonical.has_comment_marker,
        'nhanh1_label': label_name,
        'nhanh1_confidence': round(nhanh1_confidence, 4),
        'nhanh2_anomaly_score': round(anomaly_score, 4),
        'nhanh2_is_anomaly': is_anomaly,
        'verdict': verdict,
    }


## 3. Thử với vài payload mẫu

In [3]:
test_queries = [
    ("SELECT * FROM users WHERE id = 42", 'benign - SELECT thuong'),
    ("1' OR '1'='1", 'SQLi - boolean-blind kinh dien'),
    ("1' UNION SELECT username, password FROM users--", 'SQLi - union-based'),
    ("' AND SLEEP(5)--", 'SQLi - time-blind'),
    ("admin'--", 'SQLi - comment injection'),
    ("john.doe@example.com", 'benign - email nhu binh thuong'),
]

for query, note in test_queries:
    result = detect(query)
    print(f"[{note}]")
    print(f"  input: {query!r}")
    print(f"  Nhanh 1: {result['nhanh1_label']} (conf={result['nhanh1_confidence']})")
    print(f"  Nhanh 2: anomaly_score={result['nhanh2_anomaly_score']} is_anomaly={result['nhanh2_is_anomaly']}")
    print(f"  => VERDICT: {result['verdict']}")
    print()


[benign - SELECT thuong]
  input: 'SELECT * FROM users WHERE id = 42'
  Nhanh 1: normal (conf=0.954)
  Nhanh 2: anomaly_score=-4.6433 is_anomaly=False
  => VERDICT: ALLOW

[SQLi - boolean-blind kinh dien]
  input: "1' OR '1'='1"
  Nhanh 1: boolean_blind (conf=0.9995)
  Nhanh 2: anomaly_score=-4.6437 is_anomaly=False
  => VERDICT: BLOCK

[SQLi - union-based]
  input: "1' UNION SELECT username, password FROM users--"
  Nhanh 1: union_based (conf=0.6974)
  Nhanh 2: anomaly_score=-4.3742 is_anomaly=False
  => VERDICT: BLOCK

[SQLi - time-blind]
  input: "' AND SLEEP(5)--"
  Nhanh 1: time_blind (conf=1.0)
  Nhanh 2: anomaly_score=-4.6597 is_anomaly=False
  => VERDICT: BLOCK

[SQLi - comment injection]
  input: "admin'--"
  Nhanh 1: boolean_blind (conf=0.55)
  Nhanh 2: anomaly_score=-3.1409 is_anomaly=False
  => VERDICT: BLOCK

[benign - email nhu binh thuong]
  input: 'john.doe@example.com'
  Nhanh 1: normal (conf=0.9233)
  Nhanh 2: anomaly_score=-2.8913 is_anomaly=False
  => VERDICT: ALLOW

## 4. Nhập tay (thử query của riêng bạn)

Đổi giá trị `my_query` bên dưới rồi chạy cell để test.

In [4]:
my_query = "' OR 1=1 -- "  # <-- doi cho nay
detect(my_query)


{'query_raw': "' OR 1=1 -- ",
 'query_canonical': "' or 1=1 -- ",
 'has_comment_marker': 1,
 'nhanh1_label': 'boolean_blind',
 'nhanh1_confidence': 0.9997,
 'nhanh2_anomaly_score': -4.6338,
 'nhanh2_is_anomaly': False,
 'verdict': 'BLOCK'}

## 5. (Tuỳ chọn) Đánh giá nhanh trên tập test Nhánh 1

Chạy `detect()` trên một phần test set thật, so với nhãn gốc, để có con số accuracy nhanh
(không thay thế báo cáo `reports/nhanh1_eval.json` đầy đủ, chỉ để sanity-check trực quan).

In [5]:
import pandas as pd

df_test = pd.read_csv('../data/processed/nhanh1_train.csv')
df_test = df_test[df_test['split'] == 'test'].sample(20, random_state=42)

correct = 0
for _, row in df_test.iterrows():
    pred = detect(row['query_raw'])
    ok = pred['nhanh1_label'] == row['label_name']
    correct += ok
    if not ok:
        print(f"SAI: true={row['label_name']} pred={pred['nhanh1_label']} | {row['query_raw'][:60]!r}")

print(f"\nDung {correct}/{len(df_test)} tren mau 20 dong ngau nhien tu test set")


SAI: true=normal pred=boolean_blind | '/blog";sleep 15;"/wp-login.php?reauth=1&redirect_to=http://t'

Dung 19/20 tren mau 20 dong ngau nhien tu test set
